In [1]:
import pandas as pd
from pathlib import Path

bronze = Path("../data/bronze/transactions")
parts = sorted(bronze.glob("batch_date=*"))
len(parts), parts[0].name


(731, 'batch_date=2019-01-01')

In [2]:
day = pd.read_parquet(parts[0])
day.shape

(2414, 27)

In [3]:
day.dtypes


Unnamed: 0                                str
trans_date_trans_time                     str
cc_num                                    str
merchant                                  str
category                                  str
amt                                       str
first                                     str
last                                      str
gender                                    str
street                                    str
city                                      str
state                                     str
zip                                       str
lat                                       str
long                                      str
city_pop                                  str
job                                       str
dob                                       str
trans_num                                 str
unix_time                                 str
merch_lat                                 str
merch_long                        

In [4]:
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from src.silver.transactions import load_pii_key, transform_transactions

key = load_pii_key()                      # never print this
silver_day = transform_transactions(day, key)
silver_day.shape

(2414, 22)

In [5]:
print(silver_day.dtypes)
print("hash lengths:", silver_day["card_hash"].str.len().unique())
print("zip lengths:", silver_day["zip"].str.len().unique())
print("fraud_ prefix left:", silver_day["merchant"].str.startswith("fraud_").sum())
print("cards  bronze/silver:", day["cc_num"].nunique(), silver_day["card_hash"].nunique())
print("fraud  bronze/silver:", (day["is_fraud"] == "1").sum(), silver_day["is_fraud"].sum())
print("birth years:", silver_day["birth_year"].min(), silver_day["birth_year"].max())
print("deterministic:", transform_transactions(day, key)["card_hash"].equals(silver_day["card_hash"]))

trans_num                        str
trans_ts              datetime64[us]
card_hash                        str
merchant                         str
category                         str
amt                          float64
gender                           str
city                             str
state                            str
zip                              str
lat                          float64
long                         float64
city_pop                       int64
job                              str
birth_year                     int16
merch_lat                    float64
merch_long                   float64
is_fraud                        int8
_ingested_at     datetime64[us, UTC]
_source                          str
_source_file                     str
_batch_date     date32[day][pyarrow]
dtype: object
hash lengths: [64]
zip lengths: [5]
fraud_ prefix left: 0
cards  bronze/silver: 779 779
fraud  bronze/silver: 0 0
birth years: 1926 2005
deterministic: True


In [6]:
leap = [p for p in parts if p.name == "batch_date=2020-02-29"][0]
empty_bronze = pd.read_parquet(leap)
empty_silver = transform_transactions(empty_bronze, key)
print(empty_bronze.shape, empty_silver.shape)
print((empty_silver.dtypes == silver_day.dtypes).all())

(0, 27) (0, 22)
True


In [7]:
acs_root = PROJECT_ROOT / "data" / "bronze" / "census_acs"
acs_files = sorted(acs_root.rglob("*.parquet"))
print([str(f.relative_to(acs_root)) for f in acs_files])

acs = pd.read_parquet(acs_files[0])
print(acs.shape)
print(acs.dtypes)
acs.head()

['vintage=2019\\part-0.parquet']
(1, 5)
response_body                    str
_ingested_at     datetime64[us, UTC]
_source                          str
_request_url                     str
_vintage                       int32
dtype: object


,response_body,_ingested_at,_source,_request_url,_vintage
0,"[[""NAME"",""B01003_001E"",""B19013_001E"",""B01002_0...",2026-09-25 21:30:58.885064+00:00,census_acs5_api,https://api.census.gov/data/2019/acs/acs5?get=...,2019


In [8]:
import json

rows = json.loads(acs.loc[0, "response_body"])
print("rows including header:", len(rows))
print("header:", rows[0])
for r in rows[1:4]:
    print(r)

print("API key inside stored URL:", "key=" in acs.loc[0, "_request_url"])

rows including header: 33121
header: ['NAME', 'B01003_001E', 'B19013_001E', 'B01002_001E', 'state', 'zip code tabulation area']
['ZCTA5 00601', '17113', '14361', '41.9', '72', '00601']
['ZCTA5 00602', '37751', '16807', '42.9', '72', '00602']
['ZCTA5 00603', '47081', '16049', '42.1', '72', '00603']
API key inside stored URL: False


In [9]:
from src.silver.census_acs import transform_census_acs

acs_silver = transform_census_acs(acs)
print(acs_silver.shape)
print(acs_silver.dtypes)
print("unique zips:", acs_silver["zip"].is_unique)
print("missing income:", acs_silver["median_household_income"].isna().sum())
print("missing age:", acs_silver["median_age"].isna().sum())
print("population min/max:", acs_silver["population"].min(), acs_silver["population"].max())
print("income min/max:", acs_silver["median_household_income"].min(), acs_silver["median_household_income"].max())
acs_silver.head()

(33120, 8)
zip                                        str
state_fips                                 str
population                               int64
median_household_income                  Int64
median_age                             float64
_ingested_at               datetime64[us, UTC]
_source                                    str
_vintage                                 int16
dtype: object
unique zips: True
missing income: 2299
missing age: 569
population min/max: 0 128294
income min/max: 2499 250001


,zip,state_fips,population,median_household_income,median_age,_ingested_at,_source,_vintage
0,00601,72,17113,14361,41.9,2026-09-25 21:30:58.885064+00:00,census_acs5_api,2019
1,00602,72,37751,16807,42.9,2026-09-25 21:30:58.885064+00:00,census_acs5_api,2019
2,00603,72,47081,16049,42.1,2026-09-25 21:30:58.885064+00:00,census_acs5_api,2019
3,00606,72,6392,12119,44.3,2026-09-25 21:30:58.885064+00:00,census_acs5_api,2019
4,00610,72,26686,19898,42.7,2026-09-25 21:30:58.885064+00:00,census_acs5_api,2019


In [10]:
inc = acs_silver["median_household_income"]
pop0 = acs_silver["population"] == 0
print("shape:", acs_silver.shape)
print("income capped at top (250001):", (inc == 250001).sum())
print("income capped at bottom (2499):", (inc == 2499).sum())
print("population 0:", pop0.sum())
print("missing age where population 0:", acs_silver.loc[pop0, "median_age"].isna().sum())

shape: (33120, 8)
income capped at top (250001): 15
income capped at bottom (2499): 9
population 0: 344
missing age where population 0: 344
